# Class Imbalance & Model Selection

**Week 2, Day 6:** Address class imbalance with advanced techniques and select final model.

**Objectives:**
1. Understand why default threshold (0.5) is not optimal for fraud detection
2. Apply SMOTE synthetic oversampling
3. Tune decision threshold using Precision-Recall curve
4. Compare all approaches and select final model

**Reference:** project_guide.md Week 2 Day 6

## 1. Setup & Load Data

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score, precision_recall_curve, auc,
    f1_score
)

# SMOTE for oversampling
from imblearn.over_sampling import SMOTE

# XGBoost
from xgboost import XGBClassifier

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print("Libraries imported successfully")

In [ ]:
# Load processed data
processed_dir = Path('../data/processed')

X_train = joblib.load(processed_dir / 'X_train.pkl')
X_test = joblib.load(processed_dir / 'X_test.pkl')
y_train = joblib.load(processed_dir / 'y_train.pkl')
y_test = joblib.load(processed_dir / 'y_test.pkl')

# Load best XGBoost model from Day 5
xgb_baseline = joblib.load(Path('../models/xgboost_optuna.pkl'))

# Load baseline predictions for comparison
baseline_preds = joblib.load(Path('../models/baseline_predictions.pkl'))

print(f"Train: {X_train.shape[0]:,} samples")
print(f"Test: {X_test.shape[0]:,} samples")
print(f"\nFraud rate - Train: {y_train.mean():.4%}, Test: {y_test.mean():.4%}")

## 2. Understanding the Problem: Why Default Threshold Fails

### The 0.5 Threshold Problem

**Default behavior:** Models predict `fraud` if `probability > 0.5`

**Why this fails for fraud detection:**
- Fraud is **extremely rare** (0.17% of data)
- Model needs **strong evidence** to predict fraud
- Even at 50% probability, the model may be cautious

### Visualizing the Issue

Let's see what probabilities our XGBoost model actually outputs:

In [ ]:
# Get probability predictions from XGBoost
y_proba_baseline = xgb_baseline.predict_proba(X_test)[:, 1]

# Analyze probability distribution
fraud_probs = y_proba_baseline[y_test == 1]
legit_probs = y_proba_baseline[y_test == 0]

print("=== Probability Distribution Analysis ===")
print(f"\nFor Actual Fraud cases (n={len(fraud_probs)}):")
print(f"  Mean probability: {fraud_probs.mean():.4f}")
print(f"  Median probability: {np.median(fraud_probs):.4f}")
print(f"  Min probability: {fraud_probs.min():.4f}")
print(f"  Max probability: {fraud_probs.max():.4f}")

print(f"\nFor Actual Legitimate cases (n={len(legit_probs)}):")
print(f"  Mean probability: {legit_probs.mean():.4f}")
print(f"  Median probability: {np.median(legit_probs):.4f}")
print(f"  Min probability: {legit_probs.min():.4f}")
print(f"  Max probability: {legit_probs.max():.4f}")

print(f"\nAt threshold 0.5:")
pred_05 = (y_proba_baseline > 0.5).astype(int)
cm_05 = confusion_matrix(y_test, pred_05)
tn, fp, fn, tp = cm_05.ravel()
recall_05 = tp / (tp + fn)
precision_05 = tp / (tp + fp) if (tp + fp) > 0 else 0

print(f"  Fraud predictions: {pred_05.sum()}")
print(f"  Recall: {recall_05:.4%}")
print(f"  Precision: {precision_05:.4%}")
print(f"  Missed fraud cases: {fn}")

### Key Insight:

If the average fraud probability is around 0.5-0.7, using a 0.5 threshold means:
- Many fraud cases get probability 0.3-0.49 → **missed** ❌
- By **lowering the threshold**, we catch more fraud (↑ recall)
- But we also get more false alarms (↓ precision)

**This is the Precision-Recall Tradeoff we need to optimize!**

## 3. Precision-Recall Curve Analysis

The PR curve shows all possible thresholds and their resulting precision/recall.

**Goal:** Find the threshold that gives us **recall ≥ 90%** while keeping precision as high as possible.

In [ ]:
# Compute Precision-Recall curve for XGBoost
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_baseline)

# Convert to DataFrame for easier analysis
pr_df = pd.DataFrame({
    'threshold': np.append(thresholds, 1.0),
    'precision': precisions,
    'recall': recalls
})

# Find threshold that achieves 90% recall
target_recall = 0.90
optimal_threshold_idx = np.argmin(np.abs(recalls - target_recall))
optimal_threshold_90 = thresholds[optimal_threshold_idx]
optimal_precision_90 = precisions[optimal_threshold_idx]

print("=== Threshold Analysis ===")
print(f"\nTo achieve {target_recall:.0%} recall:")
print(f"  Optimal threshold: {optimal_threshold_90:.4f}")
print(f"  Expected precision: {optimal_precision_90:.4%}")

# Calculate F1 score for each threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds[best_f1_idx]

print(f"\nFor maximum F1 score:")
print(f"  Optimal threshold: {best_f1_threshold:.4f}")
print(f"  F1 score: {f1_scores[best_f1_idx]:.4f}")
print(f"  Recall: {recalls[best_f1_idx]:.4%}")
print(f"  Precision: {precisions[best_f1_idx]:.4%}")

In [ ]:
# Plot Precision-Recall Curve
plt.figure(figsize=(12, 5))

# Left: PR Curve
plt.subplot(1, 2, 1)
plt.plot(recalls, precisions, 'b-', linewidth=2, label='XGBoost')
plt.scatter(recalls[optimal_threshold_idx], precisions[optimal_threshold_idx], 
            c='red', s=100, zorder=5, label=f'90% Recall (threshold={optimal_threshold_90:.2f})')
plt.xlabel('Recall (Fraud Catch Rate)')
plt.ylabel('Precision (Correctness of Fraud Predictions)')
plt.title('Precision-Recall Curve')
plt.grid(True, alpha=0.3)
plt.legend()

# Right: Metrics vs Threshold
plt.subplot(1, 2, 2)
plt.plot(thresholds, precisions[:-1], 'b-', label='Precision', linewidth=2)
plt.plot(thresholds, recalls[:-1], 'r-', label='Recall', linewidth=2)
plt.axvline(x=0.5, color='gray', linestyle='--', label='Default (0.5)')
plt.axvline(x=optimal_threshold_90, color='green', linestyle='--', 
            label=f'90% Recall ({optimal_threshold_90:.2f})')
plt.xlabel('Decision Threshold')
plt.ylabel('Metric Value')
plt.title('Precision & Recall vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])

plt.tight_layout()
plt.savefig('../docs/images/pr_curve_analysis.png', dpi=150)
plt.show()

print("Saved: docs/images/pr_curve_analysis.png")

## 4. Approach 1: Threshold Tuning

**Simplest approach:** Keep the same model, just change the decision threshold.

**Pros:** No retraining needed, instant results
**Cons:** May not achieve all KPIs simultaneously

In [ ]:
def evaluate_at_threshold(model, X_test, y_test, threshold, model_name):
    """Evaluate model at a specific threshold."""
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > threshold).astype(int)
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_proba)
    
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"  Threshold: {threshold:.4f}")
    print(f"{'='*60}")
    print(f"\nConfusion Matrix:")
    print(f"                Predicted")
    print(f"             Legitimate  Fraud")
    print(f"Actual Legit     {tn:5d}    {fp:5d}")
    print(f"       Fraud      {fn:5d}    {tp:5d}")
    
    print(f"\nMetrics:")
    print(f"  ROC-AUC:    {roc_auc:.4f}")
    print(f"  Recall:     {recall:.4%}")
    print(f"  Precision:  {precision:.4%}")
    print(f"  F1-Score:   {f1:.4f}")
    
    return {
        'threshold': threshold,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'roc_auc': roc_auc,
        'cm': cm
    }

# Test different thresholds
print("="*70)
print("  APPROACH 1: THRESHOLD TUNING")
print("="*70)

results_threshold = {}

# Default threshold
results_threshold['default'] = evaluate_at_threshold(
    xgb_baseline, X_test, y_test, 0.5, "XGBoost (Default 0.5)"
)

# 90% Recall threshold
results_threshold['tuned_90'] = evaluate_at_threshold(
    xgb_baseline, X_test, y_test, optimal_threshold_90, f"XGBoost (Tuned {optimal_threshold_90:.4f})"
)

# Best F1 threshold
results_threshold['best_f1'] = evaluate_at_threshold(
    xgb_baseline, X_test, y_test, best_f1_threshold, f"XGBoost (Best F1 {best_f1_threshold:.4f})"
)

## 5. Approach 2: SMOTE + XGBoost

**SMOTE** = Synthetic Minority Oversampling Technique

**How it works:**
1. Take each fraud case
2. Find its k-nearest neighbors (other fraud cases)
3. Create synthetic samples along the line connecting them
4. Balance the dataset

**Important:** Apply SMOTE ONLY on training data, NOT test data!

In [ ]:
print("="*70)
print("  APPROACH 2: SMOTE + XGBoost")
print("="*70)

# Apply SMOTE to training data only
print("\nApplying SMOTE to training data...")
print(f"Original training set: {X_train.shape}")
print(f"  Fraud cases: {y_train.sum()} ({y_train.mean():.4%})")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE: {X_train_smote.shape}")
print(f"  Fraud cases: {y_train_smote.sum()} ({y_train_smote.mean():.4%})")
print(f"  Legit cases: {(y_train_smote==0).sum()}")
print(f"  Synthetic fraud created: {y_train_smote.sum() - y_train.sum():,}")

In [ ]:
# Visualize SMOTE effect (on first 2 features for illustration)
plt.figure(figsize=(12, 5))

# Before SMOTE
plt.subplot(1, 2, 1)
plt.scatter(X_train[y_train==0].iloc[:, 0], X_train[y_train==0].iloc[:, 1], 
            alpha=0.3, s=10, label='Legitimate', c='blue')
plt.scatter(X_train[y_train==1].iloc[:, 0], X_train[y_train==1].iloc[:, 1], 
            alpha=0.8, s=30, label='Fraud', c='red', marker='x')
plt.xlabel('V1 (First PCA feature)')
plt.ylabel('V2 (Second PCA feature)')
plt.title(f'Before SMOTE (n={len(X_train)})')
plt.legend()

# After SMOTE
plt.subplot(1, 2, 2)
plt.scatter(X_train_smote[y_train_smote==0].iloc[:, 0], X_train_smote[y_train_smote==0].iloc[:, 1], 
            alpha=0.1, s=5, label='Legitimate', c='blue')
# Original fraud
plt.scatter(X_train_smote[:len(y_train)][y_train_smote[:len(y_train)]==1].iloc[:, 0], 
            X_train_smote[:len(y_train)][y_train_smote[:len(y_train)]==1].iloc[:, 1], 
            alpha=0.8, s=30, label='Original Fraud', c='red', marker='x')
# Synthetic fraud
plt.scatter(X_train_smote[len(y_train):].iloc[:, 0], X_train_smote[len(y_train):].iloc[:, 1], 
            alpha=0.5, s=20, label='Synthetic Fraud', c='orange', marker='o')
plt.xlabel('V1')
plt.ylabel('V2')
plt.title(f'After SMOTE (n={len(X_train_smote)})')
plt.legend()

plt.tight_layout()
plt.savefig('../docs/images/smote_visualization.png', dpi=150)
plt.show()

print("Saved: docs/images/smote_visualization.png")

In [ ]:
# Train XGBoost on SMOTE-augmented data
print("\nTraining XGBoost on SMOTE data...")

# Load best params from Day 5
study = joblib.load(Path('../models/optuna_study.pkl'))
best_params = study.best_params.copy()
best_params.update({
    'random_state': 42,
    'n_jobs': -1,
    'eval_metric': 'logloss',
    'use_label_encoder': False,
    'scale_pos_weight': 1  # No need for weighting since data is balanced
})

xgb_smote = XGBClassifier(**best_params)
xgb_smote.fit(X_train_smote, y_train_smote)

print("Training completed!")

# Evaluate on ORIGINAL test set (not SMOTE'd!)
y_proba_smote = xgb_smote.predict_proba(X_test)[:, 1]

# Find optimal threshold for 90% recall with SMOTE model
precisions_s, recalls_s, thresholds_s = precision_recall_curve(y_test, y_proba_smote)
optimal_idx_s = np.argmin(np.abs(recalls_s - 0.90))
optimal_threshold_s = thresholds_s[optimal_idx_s]

results_smote = evaluate_at_threshold(
    xgb_smote, X_test, y_test, optimal_threshold_s, 
    f"XGBoost + SMOTE (threshold={optimal_threshold_s:.4f})"
)

## 6. Approach 3: Balanced Class Weights

Instead of SMOTE, we can tell XGBoost to pay more attention to fraud cases during training using `class_weight`.

In [ ]:
print("="*70)
print("  APPROACH 3: BALANCED CLASS WEIGHTS")
print("="*70)

# Calculate scale_pos_weight for balanced weighting
scale_pos_weight_balanced = (y_train == 0).sum() / (y_train == 1).sum()

print(f"\nCalculated scale_pos_weight: {scale_pos_weight_balanced:.2f}")
print(f"This means each fraud case is weighted as {scale_pos_weight_balanced:.0f} legit cases")

# Train with balanced weights
params_balanced = best_params.copy()
params_balanced['scale_pos_weight'] = scale_pos_weight_balanced

xgb_balanced = XGBClassifier(**params_balanced)
xgb_balanced.fit(X_train, y_train)

print("\nTraining completed!")

# Evaluate
y_proba_balanced = xgb_balanced.predict_proba(X_test)[:, 1]
precisions_b, recalls_b, thresholds_b = precision_recall_curve(y_test, y_proba_balanced)
optimal_idx_b = np.argmin(np.abs(recalls_b - 0.90))
optimal_threshold_b = thresholds_b[optimal_idx_b]

results_balanced = evaluate_at_threshold(
    xgb_balanced, X_test, y_test, optimal_threshold_b,
    f"XGBoost + Balanced Weights (threshold={optimal_threshold_b:.4f})"
)

## 7. Comprehensive Comparison

In [ ]:
# Compile all results
comparison = pd.DataFrame([
    {
        'Approach': 'XGBoost (Default 0.5)',
        'ROC-AUC': results_threshold['default']['roc_auc'],
        'Recall': results_threshold['default']['recall'],
        'Precision': results_threshold['default']['precision'],
        'F1': results_threshold['default']['f1'],
        'Threshold': 0.5
    },
    {
        'Approach': f'XGBoost (Tuned {optimal_threshold_90:.3f})',
        'ROC-AUC': results_threshold['tuned_90']['roc_auc'],
        'Recall': results_threshold['tuned_90']['recall'],
        'Precision': results_threshold['tuned_90']['precision'],
        'F1': results_threshold['tuned_90']['f1'],
        'Threshold': optimal_threshold_90
    },
    {
        'Approach': f'XGBoost + SMOTE ({optimal_threshold_s:.3f})',
        'ROC-AUC': results_smote['roc_auc'],
        'Recall': results_smote['recall'],
        'Precision': results_smote['precision'],
        'F1': results_smote['f1'],
        'Threshold': optimal_threshold_s
    },
    {
        'Approach': f'XGBoost + Balanced ({optimal_threshold_b:.3f})',
        'ROC-AUC': results_balanced['roc_auc'],
        'Recall': results_balanced['recall'],
        'Precision': results_balanced['precision'],
        'F1': results_balanced['f1'],
        'Threshold': optimal_threshold_b
    }
])

print("\n" + "="*80)
print("              COMPLETE MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

In [ ]:
# KPI Check for all approaches
print("\n" + "="*80)
print("              KPI CHECK (Target: ROC-AUC≥0.95, Recall≥0.90, Precision≥0.85)")
print("="*80)

for _, row in comparison.iterrows():
    name = row['Approach']
    auc_pass = '✅' if row['ROC-AUC'] >= 0.95 else '❌'
    recall_pass = '✅' if row['Recall'] >= 0.90 else '❌'
    precision_pass = '✅' if row['Precision'] >= 0.85 else '❌'
    
    print(f"\n{name}:")
    print(f"  ROC-AUC {auc_pass} {row['ROC-AUC']:.4f}")
    print(f"  Recall {recall_pass} {row['Recall']:.4f}")
    print(f"  Precision {precision_pass} {row['Precision']:.4f}")
    
    kpi_count = sum([row['ROC-AUC'] >= 0.95, row['Recall'] >= 0.90, row['Precision'] >= 0.85])
    print(f"  KPIs Met: {kpi_count}/3")

print("\n" + "="*80)

## 8. Final Model Selection

In [ ]:
# Select the best model based on KPI achievement and business priorities
print("\n" + "="*80)
print("              MODEL SELECTION DECISION")
print("="*80)

print("\nBusiness Priorities:")
print("  1. Catch maximum fraud (Recall ≥ 90%) - Primary KPI")
print("  2. Minimize false alarms (Precision ≥ 85%) - Secondary KPI")
print("  3. Strong discrimination (ROC-AUC ≥ 95%) - Model quality")

# Find models meeting all 3 KPIs
kpi_pass = comparison[
    (comparison['ROC-AUC'] >= 0.95) &
    (comparison['Recall'] >= 0.90) &
    (comparison['Precision'] >= 0.85)
]

if len(kpi_pass) > 0:
    print(f"\n✅ {len(kpi_pass)} model(s) meet ALL KPIs!")
    best_model = kpi_pass.iloc[0]
    print(f"\nSELECTED: {best_model['Approach']}")
    print(f"  - Recall: {best_model['Recall']:.4%}")
    print(f"  - Precision: {best_model['Precision']:.4%}")
    print(f"  - ROC-AUC: {best_model['ROC-AUC']:.4f}")
    print(f"  - Threshold: {best_model['Threshold']:.4f}")
else:
    # Find model with best recall that also has decent precision
    print("\n⚠️ No model meets all 3 KPIs. Selecting based on tradeoff...")
    
    # Prioritize recall (catch fraud) while keeping precision > 70%
    candidates = comparison[comparison['Precision'] > 0.70].sort_values('Recall', ascending=False)
    if len(candidates) > 0:
        best_model = candidates.iloc[0]
        print(f"\nSELECTED: {best_model['Approach']}")
        print(f"  - Recall: {best_model['Recall']:.4%} (Priority)")
        print(f"  - Precision: {best_model['Precision']:.4f}")
        print(f"  - ROC-AUC: {best_model['ROC-AUC']:.4f}")
        print(f"  - Threshold: {best_model['Threshold']:.4f}")
    else:
        best_model = comparison.loc[comparison['Recall'].idxmax()]

print("\n" + "="*80)

## 9. Save Final Model

In [ ]:
# Determine which model object to save based on selection
model_name = best_model['Approach']
optimal_threshold = best_model['Threshold']

if 'SMOTE' in model_name:
    final_model = xgb_smote
    technique = 'SMOTE'
elif 'Balanced' in model_name:
    final_model = xgb_balanced
    technique = 'balanced_weights'
elif 'Tuned' in model_name:
    final_model = xgb_baseline  # Same model, different threshold
    technique = 'threshold_tuning'
else:
    final_model = xgb_baseline
    technique = 'baseline'

# Save final model with metadata
models_dir = Path('../models')

joblib.dump(final_model, models_dir / 'fraud_detector_v1.pkl')

# Save model metadata
metadata = {
    'model_name': 'Fraud Detector v1.0',
    'training_date': datetime.now().isoformat(),
    'technique': technique,
    'threshold': optimal_threshold,
    'features': X_train.columns.tolist(),
    'metrics': {
        'roc_auc': float(best_model['ROC-AUC']),
        'recall': float(best_model['Recall']),
        'precision': float(best_model['Precision']),
        'f1_score': float(best_model['F1'])
    },
    'kpi_status': {
        'roc_auc_pass': best_model['ROC-AUC'] >= 0.95,
        'recall_pass': best_model['Recall'] >= 0.90,
        'precision_pass': best_model['Precision'] >= 0.85
    }
}

import json
with open(models_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\n" + "="*60)
print("  FINAL MODEL SAVED")
print("="*60)
print(f"\nModel: {models_dir / 'fraud_detector_v1.pkl'}")
print(f"Metadata: {models_dir / 'metadata.json'}")
print(f"\nTechnique: {technique}")
print(f"Optimal Threshold: {optimal_threshold:.4f}")
print(f"\nExpected Performance:")
print(f"  Recall: {best_model['Recall']:.4%}")
print(f"  Precision: {best_model['Precision']:.4%}")
print(f"  ROC-AUC: {best_model['ROC-AUC']:.4f}")
print("="*60)

## 10. Summary Report

In [ ]:
print("\n" + "="*80)
print("              DAY 6 - CLASS IMBALANCE & MODEL SELECTION SUMMARY")
print("="*80)

print("\nTechniques Evaluated:")
print("  1. Threshold Tuning - Adjust decision threshold on PR curve")
print("  2. SMOTE - Synthetic minority oversampling")
print("  3. Balanced Weights - Class-weighted training")

print("\nKey Learnings:")
print(f"  - Default 0.5 threshold is NOT optimal for imbalanced data")
print(f"  - PR curve shows precision-recall tradeoff at different thresholds")
print(f"  - SMOTE creates {y_train_smote.sum() - y_train.sum():,.0f} synthetic fraud cases")
print(f"  - Optimal threshold varies by technique ({optimal_threshold_90:.3f} to {optimal_threshold_b:.3f})")

print("\nFinal Model Selected:")
print(f"  {best_model['Approach']}")
print(f"  Threshold: {best_model['Threshold']:.4f}")

print("\nNext Steps (Day 7):")
print("  - Create model card documentation")
print("  - Generate final evaluation report")
print("  - Prepare for Week 3: API Development")

print("="*80)